<a href="https://colab.research.google.com/github/stephanie465337/Data-Science-Portfolio-C21/blob/main/NotebookLM/Module%203/3e_YAML_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with YAML

"[YAML]( https://en.wikipedia.org/wiki/YAML ) is a human-readable data-serialization language. It is commonly used for configuration files and in applications where data is being stored or transmitted."

"Originally YAML was said to mean *Yet Another Markup Language* ... it was then repurposed as *YAML Ain't Markup Language*, a recursive acronym, to distinguish its purpose as data-oriented, rather than document markup."


References:

- [Reading and parsing a YAML file with Python](https://python.land/data-processing/python-yaml)

In [ ]:
%%capture
%%bash
apt-get update
apt-get install -y jq tree

## What is YAML?

YAML is a simplified data format for serializing ( i.e. converting to a string ) data structure.  It is a superset of JSON and therefore supports the six main data types/structures of JSON:

- numbers
- strings
- boolean
- nulls
- arrays
- objects/hashes/dictionaries

YAML uses an indented syntax to represent nested objects.  YAML documents start with three dashes `---` on their own line.

Examples:
- number ( unquoted otherwise it will be interpreted as a string ):
```
1234
```
as octal ( becomes 10 decimal )
```
012
```
as hex ( becomes 18 decimal )
```
0x12
```

- string (text can be quoted or unquoted):
```
  Apple
  "Banana"
```
- boolean ( lower case ):
```
true
false
```
- nulls ( lower case ):
```
null
```
- arrays ( surrounded with square brackets ):
```
[ 1 ,2 , 3 ]
```
or each element on its own line prefixed with a dash:

  ```
  - 1
  - 2
  - 3
  ```


- objects:
```
key: value
```
or with curly braces
```
{ key: value }
```
or multi-line with indents
```
key:
    value
```

- nested objects:

  ```
  key:
    value:
    - 1
    - 2
    - 3
  ```


## Why use YAML


It's easier to read ( sometimes ) as it does not have all the curly braces that JSON has.  It is also nicely indented, which goes along with the programming style of Python.  Furthermore, it is often used for configuration, for example, Docker compose files. ( See [ELK Stack example]( https://github.com/docker/awesome-compose/blob/master/elasticsearch-logstash-kibana/compose.yaml ) ).  Lastly, it is easy ( most of the time ) to convert into JSON and back.



In [ ]:
!curl -s https://raw.githubusercontent.com/docker/awesome-compose/master/elasticsearch-logstash-kibana/compose.yaml


## Setup

In [ ]:
import yaml
import json


In [ ]:
%%bash
<<'eof' cat > config.yaml
---
rest:
  url:
    "https://example.org/primenumbers/v1"
  port:
    8443
prime_numbers:
  [2, 3, 5, 7, 11, 13, 17, 19, 23]
eof

cat -n config.yaml

In [ ]:
ls -l

## Reading and parsing (loading) a YAML file


Read YAML into a dictionary.

In [ ]:
with open('config.yaml', 'r') as file:
  prime_service = yaml.safe_load(file)
prime_service


In [ ]:
prime_service['rest']['url']


## Reading and parsing (loading) YAML strings with Python


In [ ]:
names_yaml = """
- eric
- justin
- mary-kate
- "10"
"""
names_yaml


In [ ]:
names = yaml.safe_load(names_yaml)
names


In [ ]:
type(names)

### Example

Creating a string of YAML text, assign it to a variable, and read the string into a dictionary. Notice the use of triple quotes.

In [ ]:
omlet_recipe = '''
Ingredients:
- eggs: 2
- salt: 1 tsp
- water: 1 tbsp

Directions:
- break eggs into a bowl
- wisk in salt and water
- put pan on stove at high heat
- add egg mixture to pan and cook
- put on plate when done
- eat
'''
print(omlet_recipe)


In [ ]:
omlet_dict = yaml.safe_load(omlet_recipe)
omlet_dict['Ingredients'][0]["eggs"]

## Writing (dumping) YAML to a file



In [ ]:
with open('omlet_recipe.yaml', 'w') as file:
  yaml.dump(omlet_dict, file)


In [ ]:
!cat -n omlet_recipe.yaml

## Convert YAML to JSON

In [ ]:
with open('config.yaml', 'r') as file:
  configuration = yaml.safe_load(file)
configuration


Convert the dictionary to a JSON string.

In [ ]:
config_js = json.dumps(configuration)
config_js


Convert the dictionary to a JSON file.

In [ ]:
with open('config.json', 'w') as json_file:
    json.dump(configuration, json_file)


In [ ]:
!cat -n config.json

In [ ]:
!jq . config.json

Read the JSON file into a dictionary

In [ ]:
dict_js = json.dumps(json.load(open('config.json')), indent=2)
print(dict_js)


## Convert JSON to YAML

In [ ]:
with open('config.json', 'r') as file:
    configuration = json.load(file)
configuration

In [ ]:
with open('config.v02.yaml', 'w') as yaml_file:
    yaml.dump(configuration, yaml_file)


In [ ]:
!cat -n config.v02.yaml

# Converting free text to YAML to JSON

Using the ABQ air quality data.

In [ ]:
!curl -s -O https://data.cabq.gov/airquality/aqindex/history/042222.0017


In [ ]:
!ls -l 042222.0017


Convert the first few lines of the air quality data into YAML using `sed` and save to a file.

In [ ]:
%%bash
head -30 042222.0017

In [ ]:
!tail -6 042222.0017

In [ ]:
%%bash
cat 042222.0017 |
tr -s '\r\n' '\n' |
sed -re '/,/ { s/^/  / }' |
sed -re '/,/! { s/$/:/ }' |
sed -re '{ s/,/: / }' |
sed -re '{ s/: (.*)/: "\1"/ }' |
tee aq.yaml


In [ ]:
!ls -la aq.yaml
!wc aq.yaml

In [ ]:
!cat aq.yaml


Read the YAML file into a string variable.

In [ ]:
with open('aq.yaml', 'r') as file:
  aq_yaml = file.read()
print(aq_yaml)


In [ ]:
for line in aq_yaml.split("\n"):
  if ": 0" in line:
    print(line)

Convert the string variable into a dictionary.

In [ ]:
aq_dict = yaml.safe_load(aq_yaml)
aq_dict


In [ ]:
(aq_dict['BEGIN_FILE']['TZONE']).split(",")[0]


In [ ]:
(aq_dict['BEGIN_GROUP']['INTERVAL']).split(",")[0]


Convert the dictionary to JSON.

In [ ]:
aq_json = json.dumps( aq_dict, indent = 2 )
print(aq_json)


### Finding sizes of data files from web page


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup as BS4
import requests
import re


In [ ]:
url = 'https://data.cabq.gov/airquality/aqindex/history/'
url


In [ ]:
response = requests.get( url )
response


In [ ]:
dom = BS4( response.text, "html.parser")
type(dom)

In [ ]:
pre = dom.select("pre")
len(pre)


In [ ]:
lines = str(pre[0]).split("\n")
len(lines)

In [ ]:
lines.pop()


In [ ]:
lines[-1]

In [ ]:
lines.pop(0)


In [ ]:
lines[0]

In [ ]:
len(lines)

In [ ]:
filenames =[ BS4(line, "html.parser").find("a").text for line in lines ]
filenames[:10]

In [ ]:
file_sizes = [ re.split(r' +', BS4(line, "html.parser").text)[-2] for line in lines ]
file_sizes[:10]

In [ ]:
df = pd.DataFrame(list(zip(filenames, file_sizes)))
df

In [ ]:
df[1].replace({".*K":"\1*1000",".*M":"\1*1_000_000"}, regex=True)
# .apply(eval)


In [ ]:
df["sizes"] = (
  df[1].apply(lambda x: float(x.replace("K",""))*1_000 if "K" in x else float(x.replace("M",""))*1_000_000 if "M" in x else x )
)
df



In [ ]:
# an alternative using eval() - usually not recommended
df[1].str.replace("K","*1_000").str.replace("M","*1_000_000").apply(eval).astype(int)


In [ ]:
df["sizes"].sum()/1_000_000

In [ ]:
df.columns = ["filename", "file_size_human", "file_size"]
df.head()

In [ ]:
df["url"] = "https://data.cabq.gov/airquality/aqindex/history/" + df["filename"]
df

In [ ]:
for url in df["url"][:10]:
  response = requests.get(url)
  print(url, response.text[:20])